# Day 4 · Notebook 2 — Rebuild the Agent Loop by Hand

The managed agent hides the loop. Here you rebuild it with **Converse + tools**, so you can see and control every step — and so the failure modes in notebook 4 make sense.

By the end:

- define TravelMind's tools as **real Python functions** (runnable, with an in-memory booking store)
- describe them to the model with **`toolConfig`**
- write the **ReAct loop** by hand (the rules that trip everyone up)
- run the **happy path** (valid PNR → 3 tool calls → answer)
- **reproduce the 50-loop** on purpose and watch `max_turns` save you

Runs in **us-east-1**. The tool functions run locally; the `converse` calls need AWS credentials + Claude model access (notebook 1, prereq #2).

In [ ]:
# On Colab: uncomment.  On VS Code venv: skip.
# !pip install -q boto3
import boto3, botocore, json
REGION = "us-east-1"
CLAUDE_MODEL = "us.anthropic.claude-haiku-4-5-20251001-v1:0"   # needs the 'us.' profile
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)
print("ready ·", boto3.__version__)

## 1. The data layer (from scratch)

A tool has to return *something*. In production these functions would hit your reservation system or a booking API. For the demo we use a tiny in-memory store so the loop actually runs end to end.

Note what is **missing**: there is **no `confirm_booking` / `pay` tool**. That gap is intentional — it is what we use to reproduce the runaway loop later.

In [ ]:
BOOKINGS = {
    "ABC123": {"pnr": "ABC123", "passenger": "R. Mehta", "flight": "6E-203",
               "origin": "BLR", "dest": "DEL", "status": "DISRUPTED", "disruption": "fog at DEL"},
    "ZZ999":  {"pnr": "ZZ999", "passenger": "S. Iyer", "flight": "6E-512",
               "origin": "BOM", "dest": "GOI", "status": "ON_TIME", "disruption": None},
}

def lookup_booking(pnr):
    b = BOOKINGS.get(pnr.upper())
    if not b:
        return {"found": False, "pnr": pnr}
    return {"found": True, **{k: b[k] for k in ("pnr", "passenger", "flight", "origin", "dest", "status")}}

def get_disruption_reason(pnr):
    b = BOOKINGS.get(pnr.upper())
    if not b:
        return {"found": False, "pnr": pnr}
    return {"pnr": b["pnr"], "status": b["status"], "reason": b["disruption"] or "none"}

def get_rebooking_options(pnr):
    # static options for the demo; real impl queries inventory
    return {"pnr": pnr.upper(), "options": [
        {"flight": "6E-415", "dep": "18:40", "seats": 12},
        {"flight": "6E-422", "dep": "21:10", "seats": 5},
    ]}

TOOL_IMPL = {
    "lookup_booking": lookup_booking,
    "get_disruption_reason": get_disruption_reason,
    "get_rebooking_options": get_rebooking_options,
}
print("local tools:", list(TOOL_IMPL))
print("sample:", lookup_booking("abc123"))

## 2. Describe the tools to the model — `toolConfig`

The model never sees your Python. It sees only the **name, description, and input schema**. The `description` is the *only* thing it reads to decide whether to call a tool — vague descriptions cause wrong-tool loops.

In [ ]:
def tool_spec(name, description, properties, required):
    return {"toolSpec": {
        "name": name,
        "description": description,
        "inputSchema": {"json": {"type": "object", "properties": properties, "required": required}},
    }}

TOOL_CONFIG = {"tools": [
    tool_spec("lookup_booking",
              "Look up a booking by its PNR. Returns passenger, flight, route, and status.",
              {"pnr": {"type": "string", "description": "6-character PNR"}}, ["pnr"]),
    tool_spec("get_disruption_reason",
              "Given a PNR, return why the flight was disrupted (or 'none').",
              {"pnr": {"type": "string"}}, ["pnr"]),
    tool_spec("get_rebooking_options",
              "Given a PNR, return alternative flights the passenger can be moved to.",
              {"pnr": {"type": "string"}}, ["pnr"]),
]}
print(json.dumps(TOOL_CONFIG, indent=2)[:400], "...")

## 3. The system prompt — grounded and honest

Two jobs: tell the model what it is, and forbid invention. The "never invent" line is the first (cheapest) layer of hallucination defense — we harden it in notebook 4.

In [ ]:
SYSTEM = [{"text": (
    "You are TravelMind, an airline support assistant. "
    "Use ONLY facts returned by the tools. Never invent a PNR, flight number, time, or seat count. "
    "If you cannot fulfil a request with the available tools, say so plainly. "
    "Keep answers under 80 words."
)}]
print(SYSTEM[0]["text"])

## 4. The ReAct loop, by hand

The five rules that trip everyone up, all in one function:

1. **`max_turns`** — a hard ceiling. Never run an agent loop without it.
2. After each call, **append the assistant message** (it contains the `toolUse` block). Skip this and the next call has no idea what it asked for.
3. If `stopReason == "tool_use"`, **run the tools**.
4. Return tool outputs in a **`user`** message (the Converse convention — not `assistant`, not a `tool` role).
5. **Echo `toolUseId`** back unchanged, or the API rejects it.

In [ ]:
def run_agent(user_text, max_turns=6, temperature=0.0, verbose=True):
    """Hand-built ReAct loop over Converse. Returns (answer, turns_used, messages)."""
    messages = [{"role": "user", "content": [{"text": user_text}]}]
    for turn in range(1, max_turns + 1):                          # 1) hard ceiling
        resp = bedrock_runtime.converse(
            modelId=CLAUDE_MODEL,
            messages=messages,
            system=SYSTEM,
            toolConfig=TOOL_CONFIG,
            inferenceConfig={"temperature": temperature, "maxTokens": 512},
        )
        out = resp["output"]["message"]
        messages.append(out)                                      # 2) append assistant turn
        stop = resp["stopReason"]
        if verbose:
            print(f"[turn {turn}] stopReason = {stop}")
        if stop != "tool_use":                                    # clean exit
            text = "".join(b.get("text", "") for b in out["content"])
            return text, turn, messages
        tool_results = []                                         # 3) run the tools
        for block in out["content"]:
            if "toolUse" in block:
                tu = block["toolUse"]
                name, tid, args = tu["name"], tu["toolUseId"], tu["input"]
                if verbose:
                    print(f"           -> {name}({args})")
                try:
                    result = TOOL_IMPL[name](**args)
                except Exception as e:
                    result = {"error": str(e)}
                tool_results.append({"toolResult": {
                    "toolUseId": tid,                             # 5) echo id, unchanged
                    "content": [{"json": result}],
                }})
        messages.append({"role": "user", "content": tool_results})  # 4) results in a USER msg
    return "I can't complete that. Escalating to a human agent.", max_turns, messages

## 5. Happy path — a valid, disrupted PNR

Watch the trace: the model calls `lookup_booking`, then `get_disruption_reason`, then `get_rebooking_options`, then stops with an answer. `stopReason` flips from `tool_use` to `end_turn`.

In [ ]:
answer, turns, _ = run_agent(
    "My PNR is ABC123 and my flight was disrupted. What are my options?"
)
print(f"\n--- answer (in {turns} turns) ---")
print(answer)

## 6. Inspect the steps yourself

You own `messages`, so you can read exactly what happened — the code equivalent of the console trace panel.

In [ ]:
def show_transcript(messages):
    for m in messages:
        for block in m["content"]:
            if "text" in block and block["text"].strip():
                print(f"[{m['role']}] text: {block['text'][:140]}")
            elif "toolUse" in block:
                tu = block["toolUse"]
                print(f"[{m['role']}] ACT  : {tu['name']}({tu['input']})")
            elif "toolResult" in block:
                body = block["toolResult"]["content"][0].get("json")
                print(f"[{m['role']}] OBS  : {json.dumps(body)[:140]}")

_, _, msgs = run_agent("My PNR is ABC123, what happened to my flight?", verbose=False)
show_transcript(msgs)

## 7. Reproduce the 50-loop — on purpose

Ask it to **confirm and pay** for a flight. There is no booking tool and no fallback, so the model keeps trying to *act* on a capability that does not exist. `stopReason` stays `tool_use` every turn. Only `max_turns` stops it — without that line this is an infinite, billable loop.

In [ ]:
answer, turns, _ = run_agent(
    "Confirm and pay for rebooking option 1 on PNR ABC123 right now.",
    max_turns=6,
)
print(f"\nhit the cap after {turns} turns -> {answer}")
print("\nThe model is not broken. It is obediently trying to Act with no legal move and no exit.")

### Why it looped

Three holes, the same ones we named in class: **no capability** (no booking tool), **no fallback** (nothing to do when no tool fits), **no spin guard**. Notebook 4 fixes all three — a `handoff_to_human` tool turns "confirm booking" into a legal action, so the loop ends because the model *succeeds at handing off*, not because a timer fired.

## 8. Reminder — this loop calls `bedrock-runtime` directly, so the `us.` prefix matters

The hand-built loop talks straight to the model, so the model id must be the `us.` inference profile for Claude. The cell below makes the failure shape explicit one more time.

In [ ]:
try:
    bedrock_runtime.converse(
        modelId="anthropic.claude-3-5-haiku-20241022-v1:0",   # bare id, no 'us.'
        messages=[{"role": "user", "content": [{"text": "hi"}]}],
        inferenceConfig={"maxTokens": 8},
    )
    print("bare id worked (unexpected in this account/region)")
except botocore.exceptions.ClientError as e:
    print("bare id FAILED ->", e.response["Error"]["Code"], "(use the us. profile)")

## Recap

- The loop is **yours**: `converse` → if `tool_use`, run tools, return results in a `user` message, repeat.
- Five rules: `max_turns`, append the assistant turn, results in a user message, echo `toolUseId`, exit on non-`tool_use`.
- The 50-loop reproduces with **no booking tool + no fallback** — fixed in notebook 4.

**Next — Notebook 3:** add a real **action group** to the managed agent (Lambda executor), drive an action group **from your own code** with return-control, and wire the same functions into this hand-built loop.